# Module 1: Why OpenEnv? — Your First Environments

In this notebook you'll connect to three real hosted OpenEnv environments and interact with each using the same 3-method interface: `reset()`, `step()`, `state()`.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

In [1]:
!pip install -q openenv-core fastmcp
!git clone --depth=1 -q https://github.com/meta-pytorch/OpenEnv.git 2>/dev/null || true

import sys, os
repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

Setup complete!


## 1. The Echo Environment

The simplest possible OpenEnv environment — it echoes back whatever you send. Perfect for learning the interface.

Hosted at: `https://openenv-echo-env.hf.space`

In [2]:
from envs.echo_env import EchoEnv

# EchoEnv extends MCPToolClient — it exposes tools, not raw reset/step actions.
# MCP methods (list_tools, call_tool) are async; .sync() wraps them automatically
# via SyncEnvClient.__getattr__, so the same .sync() pattern works here.
with EchoEnv(base_url='https://openenv-echo-env.hf.space').sync() as env:
    # reset() starts a new episode
    result = env.reset()
    print('After reset:')
    print(f'  Observation: {result.observation}')
    print(f'  Done: {result.done}')
    print()

    # Discover available tools
    tools = env.list_tools()
    print('Available tools:')
    for tool in tools:
        print(f'  - {tool.name}: {tool.description}')
    print()

    # call_tool() sends a message and returns the result
    response = env.call_tool('echo_message', message='Hello, OpenEnv!')
    print(f'echo_message("Hello, OpenEnv!") -> {response}')

    response = env.call_tool('echo_with_length', message='OpenEnv')
    print(f'echo_with_length("OpenEnv") -> {response}')

    # state() returns episode metadata
    state = env.state()
    print(f'\nState: step_count={state.step_count}')

ConnectionError: Failed to connect to wss://openenv-echo-env.hf.space/ws: server rejected WebSocket connection: HTTP 404

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Three methods. That's the entire API. Every OpenEnv environment works exactly like this.

## 2. OpenSpiel Catch

Now let's connect to a real game. Catch is a simple single-player game from DeepMind's OpenSpiel:

- A ball falls from the top of a 10×5 grid
- You move a paddle left/right to catch it
- Actions: `0` = left, `1` = stay, `2` = right
- Reward: `+1` if caught, `0` if missed

Same 3 methods, completely different game.

In [ ]:
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction

OPENSPIEL_URL = 'https://openenv-openspiel-catch.hf.space'

with OpenSpielEnv(base_url=OPENSPIEL_URL).sync() as env:
    result = env.reset()
    print('Game: Catch')
    print(f'Legal actions: {result.observation.legal_actions}')
    print(f'Info state length: {len(result.observation.info_state)}')
    print()

    # Play a few steps with a random policy
    import random
    step = 0
    while not result.done:
        action_id = random.choice(result.observation.legal_actions)
        action_name = {0: 'LEFT', 1: 'STAY', 2: 'RIGHT'}[action_id]
        result = env.step(OpenSpielAction(
            action_id=action_id,
            game_name='catch'
        ))
        step += 1
        print(f'Step {step}: {action_name} -> reward={result.reward}, done={result.done}')

    print(f'\nFinal reward: {result.reward}')
    state = env.state()
    print(f'State: step_count={state.step_count}')

Same pattern: `reset()` → `step()` → check `done`. The observation type is different (`OpenSpielObservation` vs `EchoObservation`), but the interface is identical.

## 3. TextArena Wordle

TextArena is a text-based game environment. Wordle gives you 6 attempts to guess a 5-letter word, with color-coded feedback after each guess.

Hosted at: `https://burtenshaw-textarena.hf.space`

In [ ]:
from envs.textarena_env import TextArenaEnv
from envs.textarena_env.models import TextArenaAction

TEXTARENA_URL = 'https://burtenshaw-textarena.hf.space'

with TextArenaEnv(base_url=TEXTARENA_URL).sync() as env:
    result = env.reset()
    print('Wordle prompt:')
    print(result.observation.prompt)
    print()

    # Make a few guesses
    guesses = ['crane', 'slate', 'blind']
    for guess in guesses:
        if result.done:
            break
        result = env.step(TextArenaAction(message=f'[{guess}]'))
        print(f'Guess: {guess}')
        for msg in result.observation.messages:
            print(f'  [{msg.category}] {msg.content}')
        print(f'  Reward: {result.reward}, Done: {result.done}')
        print()

## 4. Async vs Sync

OpenEnv clients are async by default. For notebooks and simple scripts, use the `.sync()` wrapper:

```python
# Sync (notebooks, simple scripts)
with EchoEnv(base_url=url).sync() as env:
    result = env.reset()

# Async (production, training loops)
async with EchoEnv(base_url=url) as env:
    result = await env.reset()
```

For this course, we'll use `.sync()` everywhere for simplicity.

In [ ]:
!pip install openenv


## Summary

You connected to three completely different environments — Echo, Catch, Wordle — using the same interface:

| Method | What it does |
|--------|--------------|
| `reset()` | Start a new episode |
| `step(action)` | Take an action, get observation + reward |
| `state()` | Get episode metadata |

The action and observation types change per environment, but the pattern never does.

**Next:** [Module 2](../module-2/README.md) — Using existing environments to build and compare policies.